# Clase 198 — Docker para empaquetar modelos

Notebook declarativo: genera el `Dockerfile`, `.dockerignore`, `app.py`, `requirements.txt` en un directorio temporal. Las celdas de `docker build/run` se muestran como shell commands — requieren Docker instalado para ejecutarse.

## Setup — generar el proyecto

In [ ]:
import os, shutil, tempfile
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'docker_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

# 1. Entrenar un modelo y guardarlo
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
X, y = load_iris(return_X_y=True)
m = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)
joblib.dump(m, 'model.pkl')
print('model.pkl:', Path('model.pkl').stat().st_size, 'bytes')

In [ ]:
Path('app.py').write_text('''\
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np

model = joblib.load("model.pkl")
app = FastAPI()

class In(BaseModel):
    features: list[float]

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/predict")
def predict(x: In):
    pred = int(model.predict(np.array(x.features).reshape(1, -1))[0])
    return {"class": pred}
''')

Path('requirements.txt').write_text('fastapi==0.115.0\nuvicorn[standard]==0.32.0\nscikit-learn==1.5.2\njoblib==1.4.2\n')
print('app.py + requirements.txt listos')

## 1. Dockerfile naive (mal hecho a propósito)

In [ ]:
bad = '''\
FROM python:3.12
COPY . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD uvicorn app:app --host 0.0.0.0 --port 8000
'''
Path('Dockerfile.bad').write_text(bad)
print(bad)
print('# Problemas: imagen base full (~1 GB), COPY antes de RUN pip (cache-busting),')
print('# corre como root, sin healthcheck, sin .dockerignore.')

## 2. Dockerfile multi-stage correcto

In [ ]:
good = '''\
# --- Stage 1: build wheels (incluye compiladores) ---
FROM python:3.12-slim AS builder
WORKDIR /build
RUN pip install --no-cache-dir --upgrade pip wheel
COPY requirements.txt .
RUN pip wheel --no-cache-dir --wheel-dir /wheels -r requirements.txt

# --- Stage 2: runtime slim ---
FROM python:3.12-slim AS runtime

RUN groupadd -r app && useradd -r -g app -u 1000 -m app
WORKDIR /app

COPY --from=builder /wheels /wheels
COPY requirements.txt .
RUN pip install --no-cache-dir --no-index --find-links=/wheels -r requirements.txt \\
    && rm -rf /wheels

COPY --chown=app:app app.py model.pkl ./
USER app

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1
ENTRYPOINT ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
Path('Dockerfile').write_text(good)
print(good)

## 3. `.dockerignore`

In [ ]:
Path('.dockerignore').write_text('''\
.git
.gitignore
__pycache__
*.pyc
*.ipynb
.ipynb_checkpoints
data/
mlruns/
*.md
tests/
''')
print(Path('.dockerignore').read_text())

## 4. Build, run, inspect (requiere Docker)

In [ ]:
# Comentado por default — descomentá si tenés Docker corriendo localmente.
# !docker build -t iris-api:v1 .
# !docker images iris-api:v1 --format 'TAG={{.Tag}} SIZE={{.Size}}'
# !docker run -d --name iris -p 8000:8000 iris-api:v1
# import time; time.sleep(2)
# !curl -s -X POST localhost:8000/predict -H 'content-type: application/json' -d '{"features":[5.1,3.5,1.4,0.2]}'
# !docker exec iris whoami   # debe decir "app", no "root"
# !docker history iris-api:v1 --human --format 'table {{.CreatedBy}}\t{{.Size}}'
# !docker stop iris && docker rm iris

print('Para correr esta sección, instalá Docker Desktop / Docker Engine y descomentá las líneas.')

## Ejercicio guiado

1. Buildeá `Dockerfile.bad` y `Dockerfile`. Compará tamaños (`docker images`). El multi-stage debería ser <300 MB; el naive >1 GB.
2. Cambiá una línea en `app.py` y rebuildeá ambos. Confirmá que el bueno solo reconstruye la última capa, el malo reconstruye todo.
3. Corré `trivy image iris-api:v1 --severity HIGH,CRITICAL`. Anotá las CVEs y proponé fixes (bump de base, `apt-get update`).
4. Pushá la imagen a Docker Hub (`docker push <user>/iris-api:1.0.0`) y obtené el digest. Reemplazá `:1.0.0` por `@sha256:...` en tu deploy.

## Conclusiones

- `python:3.12-slim` + multi-stage corta imagen 4× sin perder funcionalidad.
- Orden de instrucciones = velocidad de rebuild.
- Non-root + healthcheck son requisito mínimo, no "nice to have".
- Producción referencia **digest**, no `:latest`.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. `docker build/push` requiere el **daemon de Docker** (infra externa), así que los `Dockerfile` y comandos `docker` se muestran como se ejecutarían, y validamos/ilustramos el *concepto* con Python: parseamos el `Dockerfile` para chequear buenas prácticas, simulamos el **layer caching** por hash de contenido, y calculamos el **digest** (`sha256`) que hace inmutable un deploy.

In [ ]:
import hashlib, re
def parse_dockerfile(text):
    """Devuelve la lista de (instrucción, argumento) — como hace el builder."""
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split(None, 1)
        out.append((parts[0].upper(), parts[1] if len(parts) > 1 else ''))
    return out
print('parser de Dockerfile listo.')

### Ejercicio 1 — Dockerfile básico

`FROM python:3.12-slim`, instalar deps, `COPY`, `CMD`. Validamos que las instrucciones esperadas están presentes y en orden razonable (deps antes de copiar el código, para cache).

In [ ]:
dockerfile_v1 = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . /app
CMD ["python", "predict.py"]
'''
instrs = parse_dockerfile(dockerfile_v1)
kinds = [k for k, _ in instrs]
assert kinds[0] == 'FROM', 'un Dockerfile empieza con FROM'
assert 'CMD' in kinds and 'RUN' in kinds
# buena práctica: requirements se copian ANTES que el resto del código (mejor cache)
copy_req = next(i for i, (k, a) in enumerate(instrs) if k == 'COPY' and 'requirements' in a)
run_pip  = next(i for i, (k, a) in enumerate(instrs) if k == 'RUN' and 'pip install' in a)
copy_all = next(i for i, (k, a) in enumerate(instrs) if k == 'COPY' and a.startswith('.'))
assert copy_req < run_pip < copy_all, 'orden subóptimo para el cache'
print('CLI: docker build -t miml:v1 . && docker run --rm miml:v1')
print('OK — Dockerfile válido, orden de capas cache-friendly.')

### Ejercicio 2 — Layer caching

Docker cachea una capa por el hash de sus inputs. Si cambiás `predict.py` pero no `requirements.txt`, la capa de `pip install` se **reusa** (CACHED). Lo simulamos hasheando el contenido que define cada capa.

In [ ]:
def layer_hash(*inputs):
    return hashlib.sha256('||'.join(inputs).encode()).hexdigest()[:12]

req = 'scikit-learn==1.5.0\nnumpy'
# build 1
h_pip_1  = layer_hash('RUN pip install', req)
h_code_1 = layer_hash('COPY .', 'predict.py:v1')
# build 2: cambia predict.py, NO requirements.txt
h_pip_2  = layer_hash('RUN pip install', req)
h_code_2 = layer_hash('COPY .', 'predict.py:v2')

print(f'pip layer : {h_pip_1} -> {h_pip_2}  {"CACHED" if h_pip_1 == h_pip_2 else "REBUILD"}')
print(f'code layer: {h_code_1} -> {h_code_2}  {"CACHED" if h_code_1 == h_code_2 else "REBUILD"}')
assert h_pip_1 == h_pip_2, 'requirements no cambió => pip layer CACHED'
assert h_code_1 != h_code_2
print('OK — por eso `COPY requirements.txt` va ANTES: cambiar código no reinstala deps.')

### Ejercicio 3 — Multi-stage build

`builder` compila deps con toolchain pesado; `runtime` (slim) copia solo `site-packages`. Resultado: imagen final mucho más chica. El YAML es declarativo; ilustramos la reducción de tamaño esperada.

In [ ]:
dockerfile_multi = '''
FROM python:3.12 AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/deps -r requirements.txt

FROM python:3.12-slim AS runtime
COPY --from=builder /deps /usr/local/lib/python3.12/site-packages
COPY . /app
WORKDIR /app
CMD ["python", "predict.py"]
'''
stages = [a.split(' AS ')[-1] for k, a in parse_dockerfile(dockerfile_multi) if k == 'FROM']
assert stages == ['builder', 'runtime'], 'deben existir 2 stages'
# ilustración de tamaños típicos (MB): python:3.12 (~1000) vs python:3.12-slim (~130) + deps
sizes = {'single (python:3.12)': 1000 + 250, 'multi (slim + deps)': 130 + 250}
for k, v in sizes.items():
    print(f'{k:28} ~{v} MB')
assert sizes['multi (slim + deps)'] < sizes['single (python:3.12)']
print('OK — multi-stage descarta compiladores/headers => imagen final más liviana.')

### Ejercicio 4 — Usuario non-root

`RUN useradd -m app && USER app` antes del `CMD`. `docker run --rm miml:v3 whoami` debe imprimir `app`. Verificamos que el Dockerfile deja de correr como root.

In [ ]:
dockerfile_nonroot = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . /app
RUN useradd -m app
USER app
CMD ["python", "predict.py"]
'''
instrs = parse_dockerfile(dockerfile_nonroot)
user_idx = [i for i, (k, _) in enumerate(instrs) if k == 'USER']
cmd_idx = next(i for i, (k, _) in enumerate(instrs) if k == 'CMD')
assert user_idx and user_idx[-1] < cmd_idx, 'USER debe estar antes del CMD'
effective_user = instrs[user_idx[-1]][1]
print('usuario efectivo del contenedor:', effective_user, '(docker run ... whoami => app)')
assert effective_user == 'app'
print('OK — corre non-root: si un exploit escapa al proceso, no es root en el contenedor.')

### Ejercicio 5 — Tags mutables vs digest inmutable

Un tag (`miml:v1`) es un puntero **mutable**; el digest (`sha256:...`) identifica el contenido exacto y es **inmutable**. Producción referencia el digest para que un `docker push` posterior con el mismo tag no cambie lo desplegado. Calculamos un digest a partir del "manifest".

In [ ]:
# CLI real:
#   docker push miml:v1
#   docker inspect miml:v1 --format '{{index .RepoDigests 0}}'
def image_digest(layer_hashes, config):
    manifest = json.dumps({'layers': layer_hashes, 'config': config}, sort_keys=True)
    return 'sha256:' + hashlib.sha256(manifest.encode()).hexdigest()

import json
digest_v1 = image_digest([h_pip_1, h_code_1], {'cmd': ['python', 'predict.py']})
# alguien re-pushea miml:v1 con código distinto -> MISMO tag, DISTINTO digest
digest_v1b = image_digest([h_pip_1, h_code_2], {'cmd': ['python', 'predict.py']})
print('tag miml:v1 hoy   ->', digest_v1[:24], '...')
print('tag miml:v1 mañana->', digest_v1b[:24], '...')
assert digest_v1 != digest_v1b
print('OK — el tag es mutable; deployá por digest (miml@sha256:...) para reproducibilidad.')